<div style="border-left: 5px solid #1f4e79; padding: 16px 20px; background-color: #f8fafc; border: 1px solid #d9e2ec; border-radius: 6px; margin-bottom: 24px; font-family: Arial, sans-serif;">
  <h1 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 28px;">
  <p style="margin: 0; color: #4b5563; font-size: 15px; line-height: 1.55;">Notebook 10 — Sélection des features et scénarios de modélisation Construction du référentiel de variables admissibles et des scénarios de comparaison pour la phase de modélisation. Ce notebook transforme la base enrichie issue du notebook 09 en une base de modélisation documentée. Il ne crée pas encore de modèle : il définit les familles de variables, écarte les champs à risque de fuite de cible, rattache les variables de taux si nécessaire et formalise les scénarios qui seront comparés ensuite.</p>
</div>

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Objectif</h3>
  <div style="color: #4b5563; font-size: 14px;">Rôle du notebook dans le pipeline L’objectif est de passer d’un jeu de données très large à une structure exploitable pour la modélisation. La sélection est volontairement séparée de l’entraînement afin de rendre les choix de variables auditables, reproductibles et comparables d’un scénario à l’autre. Identifier la cible disponible pour la modélisation.  Classer les variables par familles métier : bien, temps, territoire, socio-économie, comparables et taux.  Construire des scénarios progressifs afin de mesurer l’apport marginal de chaque famille.</div>
</div>

In [ ]:
# Import
from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("default")

PROJECT_ROOT = Path(
    r"C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE"
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Entrée canonique issue du notebook 09.
INPUT_PATH = DATA_PROCESSED / "df_biens_residentiels_features_comparables.parquet"

# Source complémentaire issue du notebook 07 pour créer les features de taux si absentes de l'entrée 09.
TAUX_SOURCE_PATH = DATA_PROCESSED / "df_biens_residentiels_temporel.parquet"

# Sorties canoniques du notebook 10.
OUTPUT_PATH = DATA_PROCESSED / "df_modelisation_scenarios_features.parquet"
OUTPUT_METADATA_PATH = DATA_PROCESSED / "df_scenarios_features_metadata.parquet"
OUTPUT_FEATURE_LISTS_PATH = DATA_PROCESSED / "dict_scenarios_features.json"

SELECTION_DIR = OUTPUTS_DIR / "selection_features_scenarios"
REPORTS_DIR = SELECTION_DIR / "reports"
CONFIG_DIR = SELECTION_DIR / "config"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATE = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Projet :", PROJECT_ROOT)
print("Date exécution :", RUN_DATE)
print("Entrée notebook 10 :", INPUT_PATH)
print("Source taux :", TAUX_SOURCE_PATH)
print("Sortie base scénarios :", OUTPUT_PATH)
print("Sortie métadonnées :", OUTPUT_METADATA_PATH)
print("Sortie dictionnaire scénarios :", OUTPUT_FEATURE_LISTS_PATH)

Projet : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE
Date exécution : 2026-05-25 11:16:36
Entrée notebook 10 : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_biens_residentiels_features_comparables.parquet
Source taux : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_biens_residentiels_temporel.parquet
Sortie base scénarios : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_modelisation_scenarios_features.parquet
Sortie métadonnées : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_scenarios_features_metadata.parquet
Sortie dictionnaire scénarios : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\dict_scenarios_features.json


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;
  
  <div style="color: #4b5563; font-size: 14px;">Fonctions utilitaires et contrôles Les fonctions communes standardisent les contrôles de répertoire, la normalisation des clés et le calcul des indicateurs de qualité. Elles évitent de disperser la logique technique dans les étapes analytiques. Création sécurisée des dossiers si nécessaire.  Normalisation minimale des colonnes clés.  Contrôles de valeurs manquantes et de typage numérique.</div>
</div>

In [ ]:
# Fonctions utilitaires
def ensure_dir(path: Path) -> Path:
    """
    Crée un dossier s'il n'existe pas déjà.
    """

    # Création récursive des dossiers
    path.mkdir(
        parents=True,
        exist_ok=True
    )

    return path


def first_existing(paths):
    """
    Retourne le premier chemin existant
    dans une liste de chemins.
    """

    existing = [
        Path(p)
        for p in paths
        if Path(p).exists()
    ]

    return existing[0] if existing else None


def read_first_existing(
    paths,
    label: str,
    required: bool = True
):
    """
    Charge le premier fichier parquet trouvé.
    """

    # Recherche du premier fichier existant
    path = first_existing(paths)

    # Aucun fichier trouvé
    if path is None:

        msg = (
            f"Aucun fichier trouvé pour {label}.\n"
            "Fichiers testés :\n"
            + "\n".join(f"- {p}" for p in paths)
        )

        # Erreur bloquante si le fichier est obligatoire
        if required:
            raise FileNotFoundError(msg)

        # Warning simple sinon
        warnings.warn(msg)

        return None, None

    # Lecture du fichier parquet
    df = pd.read_parquet(path)

    # Informations de contrôle
    print(f"{label} chargé : {path}")

    print(
        f"Shape {label} : "
        f"{df.shape[0]:,} lignes | "
        f"{df.shape[1]:,} colonnes"
    )

    return df, path


def normalize_keys(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise les clés territoriales
    pour éviter les problèmes de merge.
    """

    df = df.copy()

    # Normalisation du code IRIS
    if "code_iris" in df.columns:

        df["code_iris"] = (
            df["code_iris"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(9)
        )

    # Normalisation du code commune
    if "code_commune" in df.columns:

        df["code_commune"] = (
            df["code_commune"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(5)
        )

    # Normalisation du code département
    if "code_departement" in df.columns:

        df["code_departement"] = (
            df["code_departement"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(2)
        )

    return df


def safe_merge_m1(
    base: pd.DataFrame,
    enrich: pd.DataFrame,
    key: str,
    label: str
) -> pd.DataFrame:
    """
    Réalise un merge sécurisé de type many-to-one.
    """

    # Vérification table vide
    if enrich is None or enrich.empty:

        warnings.warn(
            f"Table {label} vide : merge ignoré."
        )

        return base

    # Vérification présence clé dans la base
    if key not in base.columns:

        warnings.warn(
            f"Clé {key} absente de la base principale : "
            f"merge {label} ignoré."
        )

        return base

    # Vérification présence clé dans la table enrichie
    if key not in enrich.columns:

        warnings.warn(
            f"Clé {key} absente de {label} : merge ignoré."
        )

        return base

    # Normalisation des clés
    base = normalize_keys(base)
    enrich = normalize_keys(enrich)

    # Sauvegarde des colonnes initiales
    before_cols = set(base.columns)

    # Suppression des doublons sur la clé
    enrich = enrich.drop_duplicates(key)

    # Colonnes réellement nouvelles
    cols_to_add = [
        c
        for c in enrich.columns
        if c == key or c not in base.columns
    ]

    # Merge sécurisé many-to-one
    out = base.merge(
        enrich[cols_to_add],
        on=key,
        how="left",
        validate="m:1"
    )

    # Liste des colonnes ajoutées
    added = [
        c
        for c in out.columns
        if c not in before_cols
    ]

    print(
        f"Merge {label} sur {key} : "
        f"+{len(added)} colonnes"
    )

    # Aperçu des colonnes ajoutées
    if added:
        print(
            "Colonnes ajoutées exemple :",
            added[:25]
        )

    return out


def pct_missing(s: pd.Series) -> float:
    """
    Calcule le pourcentage
    de valeurs manquantes.
    """

    return float(s.isna().mean())


def is_numeric(s: pd.Series) -> bool:
    """
    Vérifie si une série
    est de type numérique.
    """

    return pd.api.types.is_numeric_dtype(s)


In [ ]:
 # Chargement de la base issue du notebook 09
 
EXPECTED_MIN_ROWS = 3_500_000
STRICT_ROW_CONTROL = False

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Fichier d'entrée introuvable : {INPUT_PATH}\n"
        "Exécuter le notebook 09 ou rapatrier son output canonique."
    )

df = pd.read_parquet(INPUT_PATH).copy()
df = normalize_keys(df)

if df.shape[0] < EXPECTED_MIN_ROWS:
    msg = (
        f"Volumétrie sous contrôle : {df.shape[0]:,} lignes < {EXPECTED_MIN_ROWS:,}. "
        "Vérifier le pointage avant modélisation."
    )
    if STRICT_ROW_CONTROL:
        raise ValueError(msg)
    warnings.warn(msg)

print("Fichier chargé :", INPUT_PATH)
print("Shape base notebook 09 :", df.shape)
print("Colonnes geo_* détectées :", len([c for c in df.columns if str(c).startswith("geo_")]))
print("Colonnes marché détectées :", [
    c for c in [
        "typologie_metier",
        "cluster_kmeans",
        "score_liquidite_marche",
        "score_tension_marche"
    ]
    if c in df.columns
])

 
# Création / rattachement des features de taux
features_taux_credit_reference = [
    "taux_credit_moyen",
    "variation_taux_1m",
    "variation_taux_3m",
    "taux_credit_roll3",
    "taux_credit_lag1",
    "taux_credit_lag3",
    "score_tension_credit",
]

features_taux_credit = [
    col for col in features_taux_credit_reference
    if col in df.columns
]

if not features_taux_credit:
    if not TAUX_SOURCE_PATH.exists():
        raise FileNotFoundError(
            f"Fichier temporel enrichi introuvable : {TAUX_SOURCE_PATH}"
        )

    df_taux_source = pd.read_parquet(TAUX_SOURCE_PATH)

    features_taux_disponibles = [
        col for col in features_taux_credit_reference
        if col in df_taux_source.columns
    ]

    if not features_taux_disponibles:
        raise ValueError(
            "Le fichier temporel enrichi ne contient aucune feature de taux attendue."
        )

    if "id_mutation" in df.columns and "id_mutation" in df_taux_source.columns:
        df_taux_join = (
            df_taux_source[["id_mutation"] + features_taux_disponibles]
            .drop_duplicates("id_mutation")
            .copy()
        )

        df = df.merge(
            df_taux_join,
            on="id_mutation",
            how="left",
            validate="many_to_one"
        )

        print("Features de taux créées par rattachement via id_mutation.")

    elif "annee_mois" in df.columns and "annee_mois" in df_taux_source.columns:
        df["annee_mois"] = df["annee_mois"].astype("string")
        df_taux_source["annee_mois"] = df_taux_source["annee_mois"].astype("string")

        df_taux_join = (
            df_taux_source[["annee_mois"] + features_taux_disponibles]
            .drop_duplicates("annee_mois")
            .copy()
        )

        df = df.merge(
            df_taux_join,
            on="annee_mois",
            how="left",
            validate="many_to_one"
        )

        print("Features de taux créées par rattachement via annee_mois.")

    else:
        raise ValueError(
            "Impossible de créer les features de taux : aucune clé commune id_mutation ou annee_mois."
        )

features_taux_credit = [
    col for col in features_taux_credit_reference
    if col in df.columns
]

if not features_taux_credit:
    raise ValueError("Aucune feature de taux n'a été créée.")

display(pd.DataFrame({"features_taux_credit": features_taux_credit}))

display(
    df[features_taux_credit]
    .isna()
    .mean()
    .sort_values()
    .to_frame("part_valeurs_manquantes")
)

print("Shape après rattachement des taux :", df.shape)

Fichier chargé : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_biens_residentiels_features_comparables.parquet
Shape base notebook 09 : (3763971, 397)
Colonnes geo_* détectées : 148
Colonnes marché détectées : ['score_liquidite_marche']
Features de taux créées par rattachement via annee_mois.


,features_taux_credit
0,taux_credit_moyen
1,variation_taux_1m
2,variation_taux_3m
3,taux_credit_roll3
4,taux_credit_lag1
5,taux_credit_lag3
6,score_tension_credit


,part_valeurs_manquantes
taux_credit_moyen,0.0000
variation_taux_1m,0.0000
variation_taux_3m,0.0000
taux_credit_roll3,0.0000
taux_credit_lag1,0.0000
taux_credit_lag3,0.0000
score_tension_credit,0.0000


Shape après rattachement des taux : (3763971, 404)


In [ ]:
# Référentiels de variables par famille.
# Cette cellule définit les blocs de features disponibles
# et la sélection prudente utilisée pour les scénarios de modélisation.

TARGET_CANDIDATES = [
    "log_prix_m2",
    "prix_m2_cible",
    "prix_m2",
]

# Détection automatique de la cible disponible dans le dataframe.
TARGET = next(
    (c for c in TARGET_CANDIDATES if c in df.columns),
    None,
)

print("Cible détectée :", TARGET)


# Colonnes à exclure des features explicatives.
# Elles correspondent à la cible, à des transformations directes du prix,
# à la valeur foncière ou à des identifiants techniques.
LEAKAGE_OR_TARGET_COLUMNS = {
    "prix_m2",
    "log_prix_m2",
    "prix_m2_model",
    "prix_m2_cible",
    "prix_m2_winsor",
    "prix_m2_winsor_01_99",
    "valeur_fonciere",
    "log_valeur_fonciere",
    "valeur_fonciere_model",
    "id_mutation",
    "id_parcelle",
    "adresse",
    "numero_disposition",
}

# Motifs identifiant des variables construites directement
# à partir du prix courant.
LEAKAGE_REGEX = re.compile(
    r"^(ratio_prix_vs_|ecart_abs_prix_vs_|ecart_pct_vs_)"
)

# Codes géographiques conservés comme variables de contexte.
GEO_PROXY_COLUMNS = {
    "code_departement",
    "code_commune",
    "code_iris",
}


# Colonnes issues du notebook B3 marché.
# Elles décrivent la structure territoriale du marché.
B3_MARKET_COLUMNS = [
    "signal_marche_fiable",
    "score_liquidite_marche",
    "score_tension_marche",
    "score_premium_marche",
    "score_atypicite_marche",
    "typologie_metier",
    "typologie_metier_code",
    "cluster_kmeans",
    "cluster_kmeans_code",
    "distance_centroid_cluster",
    "transactions_count",
    "transactions_par_mois",
    "dispersion_prix_relative",
    "pca_1",
    "pca_2",
]

# Sous-ensemble B3 plus prudent.
# Les variables directement liées aux prix agrégés sont évitées.
B3_MARKET_SELECTED_CORE = [
    "signal_marche_fiable",
    "score_liquidite_marche",
    "score_atypicite_marche",
    "transactions_count",
    "transactions_par_mois",
    "typologie_metier_code",
    "cluster_kmeans_code",
    "distance_centroid_cluster",
]


def is_safe_b5_column(col: str) -> bool:
    """
    Identifie les variables B5 comparables admissibles.

    Sont exclues :
    - les variables directement dérivées du prix courant ;
    - la cible ;
    - la valeur foncière ;
    - les ratios ou écarts calculés contre la cible.
    """

    c = str(col)

    forbidden = [
        r"^ratio_prix_vs_",
        r"^ecart_abs_prix_vs_",
        r"^ecart_pct_vs_",
        r"^prix_m2_cible$",
        r"^prix_m2$",
        r"^log_prix_m2$",
        r"^valeur_fonciere$",
    ]

    if any(re.search(p, c) for p in forbidden):
        return False

    # Les variables géographiques geo_* sont conservées.
    # Elles correspondent à des comparables locaux passés.
    if c.startswith("geo_"):
        return True

    # Agrégats historiques passés admissibles.
    if c in {
        "commune_prix_m2_med_passe",
        "commune_type_prix_m2_med_passe",
        "iris_prix_m2_med_passe",
        "commune_prix_m2_iqr_passe",
        "iris_prix_m2_iqr_passe",
        "commune_nb_ventes_passe",
        "iris_nb_ventes_passe",
        "commune_type_nb_ventes_passe",
    }:
        return True

    # Colonnes contenant une logique de comparables,
    # sous réserve qu'elles ne soient pas dérivées de la cible.
    if re.search(r"(comparable|comparables)", c, flags=re.I):
        return not any(
            w in c.lower()
            for w in [
                "ratio",
                "ecart",
                "target",
                "cible",
                "valeur_fonciere",
            ]
        )

    return False


# Liste complète des variables B5 admissibles.
B5_CANDIDATES = sorted(
    [c for c in df.columns if is_safe_b5_column(c)]
)


def select_b5_recommended(cols: list[str]) -> list[str]:
    """
    Sélectionne un sous-ensemble B5 recommandé.

    Objectif :
    conserver des variables de comparables proches,
    interprétables et non directement dérivées de la cible.
    """

    selected = []

    # Agrégats historiques non géographiques prioritaires.
    preferred_exact = [
        "commune_prix_m2_med_passe",
        "commune_type_prix_m2_med_passe",
        "iris_prix_m2_med_passe",
        "commune_prix_m2_iqr_passe",
        "iris_prix_m2_iqr_passe",
        "commune_nb_ventes_passe",
        "iris_nb_ventes_passe",
        "commune_type_nb_ventes_passe",
    ]

    selected.extend(
        [c for c in preferred_exact if c in cols]
    )

    # Fenêtres métier privilégiées.
    # On cible des rayons proches et des fenêtres temporelles raisonnables.
    radii = [300, 500, 1000]
    months = [12, 24]

    metrics_priority = [
        "prix_m2_med",
        "prix_m2_iqr",
        "prix_m2_q25",
        "prix_m2_q75",
        "nb",
        "count",
        "dist_med",
        "distance_med",
        "surface_med",
    ]

    for r in radii:
        for m in months:
            for metric in metrics_priority:

                # Priorité aux comparables proches en surface,
                # puis aux variantes plus générales.
                patterns = [
                    rf"^geo_{r}m_{m}m_surface_20_.*{metric}",
                    rf"^geo_{r}m_{m}m_.*{metric}",
                ]

                matches = [
                    c for c in cols
                    if any(re.search(p, c) for p in patterns)
                ]

                # Une seule colonne par combinaison rayon / fenêtre / métrique
                # afin de limiter la redondance.
                if matches:
                    selected.append(sorted(matches)[0])

    # Fallback si aucune règle ne matche.
    if not selected:
        selected = [
            c for c in cols
            if any(
                k in c
                for k in [
                    "prix_m2_med",
                    "prix_m2_iqr",
                    "nb",
                    "dist_med",
                ]
            )
        ]

    # Suppression des doublons en conservant l'ordre.
    return list(
        dict.fromkeys(
            [c for c in selected if c in cols]
        )
    )


# Sous-ensemble recommandé de variables B5.
B5_SELECTED_RECOMMENDED = select_b5_recommended(
    B5_CANDIDATES
)


# Référentiel large des variables candidates par bloc.
FEATURE_BLOCKS_CANDIDATES = {
    "B1_bien": [
        "surface_reference",
        "surface_reference_model",
        "log_surface_reference",
        "classe_surface",
        "nombre_pieces_principales",
        "nb_pieces_total",
        "surface_par_piece",
        "type_local",
        "type_bien",
        "type_bien_residentiel",
        "segment_residentiel_surface",
        "is_maison",
        "is_appartement",
        "surface_terrain",
        "has_terrain",
        "ratio_terrain_bati",
        "intensite_batie",
        "terrain_info_disponible",
        "has_dependance",
        "nb_dependances",
        "pieces_info_disponible",
        "dependance_info_disponible",
    ],

    "B2_temps": [
        "annee_mutation",
        "annee",
        "trimestre_mutation",
        "trimestre",
        "mois_mutation",
        "mois",
        "mois_sin",
        "mois_cos",
        "annee_mois_mutation",
        "mois_depuis_debut",
        "prix_med_lag1",
        "prix_m2_median_lag1",
        "volume_tx_lag1",
        "nb_transactions_lag1",
        "variation_prix_3m_lag1",
        "variation_prix_lag1",
        "variation_volume_3m_lag1",
        "variation_volume_lag1",
        "momentum_prix_3_6_lag1",
        "volatilite_prix_6m_lag1",
        "score_momentum_lag1",
        "score_tension_temporelle_lag1",
        "score_risque_temporel_lag1",
        "indice_saisonnier_prix",
        "indice_saisonnier_volume",
    ],

    "B3_territoire": (
        ["code_departement", "code_commune", "code_iris"]
        + B3_MARKET_COLUMNS
    ),

    "B4_socio_eco": [
        "revenu_median",
        "filosofi_revenu_median",
        "revenu_q1",
        "revenu_q3",
        "indice_inegalite_revenus",
        "taux_pauvrete",
        "part_chomage_revenu_disponible",
        "population",
        "population_2021",
        "densite_population",
        "densite_population_km2",
        "part_population_15_29",
        "part_population_65_plus",
        "taille_menage_moyenne",
        "nb_logements_2022",
        "densite_logements",
        "densite_logements_km2",
        "nb_logements_vacants",
        "part_logements_vacants",
        "logement_part_vacance",
        "logement_part_residences_secondaires",
        "part_residences_secondaires",
        "part_logements_sociaux",
    ],

    "B5_comparables": B5_CANDIDATES,
}


# Sélection métier recommandée.
# Cette version est plus prudente et destinée aux scénarios de modélisation.
FEATURE_BLOCKS_SELECTED_REFERENCE = {
    "B1_bien": [
        "surface_reference_model",
        "surface_reference",
        "log_surface_reference",
        "classe_surface",
        "nb_pieces_total",
        "surface_par_piece",
        "type_bien",
        "type_bien_residentiel",
        "segment_residentiel_surface",
        "is_maison",
        "is_appartement",
        "surface_terrain",
        "has_terrain",
        "ratio_terrain_bati",
        "intensite_batie",
        "terrain_info_disponible",
        "has_dependance",
        "nb_dependances",
        "pieces_info_disponible",
        "dependance_info_disponible",
    ],

    "B2_temps": [
        "annee_mutation",
        "annee",
        "mois_sin",
        "mois_cos",
        "trimestre_mutation",
        "trimestre",
        "mois_depuis_debut",
        "prix_m2_median_lag1",
        "volume_tx_lag1",
        "nb_transactions_lag1",
        "variation_prix_3m_lag1",
        "variation_volume_3m_lag1",
        "momentum_prix_3_6_lag1",
        "volatilite_prix_6m_lag1",
        "score_momentum_lag1",
        "score_tension_temporelle_lag1",
        "score_risque_temporel_lag1",
        "indice_saisonnier_prix",
        "indice_saisonnier_volume",
    ],

    "B3_territoire": (
        ["code_departement", "code_commune", "code_iris"]
        + B3_MARKET_SELECTED_CORE
    ),

    "B4_socio_eco": [
        "revenu_median",
        "revenu_q1",
        "revenu_q3",
        "indice_inegalite_revenus",
        "part_chomage_revenu_disponible",
        "densite_population_km2",
        "part_population_15_29",
        "part_population_65_plus",
        "taille_menage_moyenne",
        "densite_logements_km2",
        "part_logements_vacants",
        "part_residences_secondaires",
        "part_logements_sociaux",
    ],

    "B5_comparables": B5_SELECTED_RECOMMENDED,
}


# Ajout du bloc B6 relatif aux taux de crédit.
# Cette famille permet de tester l'apport du contexte de financement.
FEATURE_BLOCKS_CANDIDATES["B6_taux_credit"] = features_taux_credit
FEATURE_BLOCKS_SELECTED_REFERENCE["B6_taux_credit"] = features_taux_credit

print("Famille B6_taux_credit ajoutée :", features_taux_credit)

Cible détectée : log_prix_m2
Famille B6_taux_credit ajoutée : ['taux_credit_moyen', 'variation_taux_1m', 'variation_taux_3m', 'taux_credit_roll3', 'taux_credit_lag1', 'taux_credit_lag3', 'score_tension_credit']


In [ ]:
# Audit de disponibilité et qualité des variables.

# Seuil maximal de valeurs manquantes admissible.
# Ce seuil est volontairement large à ce stade :
# il sert à écarter uniquement les variables quasi vides.
MAX_MISSING_RATE = 0.98

# Nombre minimal de valeurs distinctes.
# Une variable constante n'apporte pas de signal exploitable.
MIN_NUNIQUE = 2


def variable_status(fam: str, var: str) -> dict:
    """
    Retourne les informations de qualité d'une variable.

    La fonction vérifie :
    - sa présence dans la base ;
    - son type ;
    - son taux de valeurs manquantes ;
    - sa cardinalité ;
    - son risque de leakage ;
    - son admissibilité minimale.
    """

    present = var in df.columns

    # Variable absente : on conserve une ligne d'audit
    # pour tracer les écarts entre référentiel et données réelles.
    if not present:
        is_leak = (
            var in LEAKAGE_OR_TARGET_COLUMNS
        ) or bool(
            LEAKAGE_REGEX.search(var)
        )

        return {
            "famille": fam,
            "variable": var,
            "presente": False,
            "dtype": None,
            "pct_manquant": np.nan,
            "nunique": 0,
            "is_numeric": False,
            "is_leakage_or_target": is_leak,
            "is_geo_proxy": var in GEO_PROXY_COLUMNS,
            "admissible_base": False,
        }

    s = df[var]

    # Mesures de qualité de base.
    miss = pct_missing(s)
    nun = int(s.nunique(dropna=True))

    # Détection des variables interdites :
    # cible, prix direct, valeur foncière ou ratio dérivé de la cible.
    is_leak = (
        var in LEAKAGE_OR_TARGET_COLUMNS
    ) or bool(
        LEAKAGE_REGEX.search(var)
    )

    # Critère minimal d'admissibilité.
    # Cette étape ne constitue pas encore une sélection modèle :
    # elle filtre seulement les variables inutilisables ou risquées.
    admissible = (
        not is_leak
        and miss <= MAX_MISSING_RATE
        and nun >= MIN_NUNIQUE
    )

    return {
        "famille": fam,
        "variable": var,
        "presente": True,
        "dtype": str(s.dtype),
        "pct_manquant": miss,
        "nunique": nun,
        "is_numeric": is_numeric(s),
        "is_leakage_or_target": is_leak,
        "is_geo_proxy": var in GEO_PROXY_COLUMNS,
        "admissible_base": admissible,
    }


# Construction de l'audit pour l'ensemble des familles de variables.
candidate_rows = []

for fam, vars_ in FEATURE_BLOCKS_CANDIDATES.items():
    for v in vars_:
        candidate_rows.append(
            variable_status(fam, v)
        )


audit_candidates = (
    pd.DataFrame(candidate_rows)
    .drop_duplicates(["famille", "variable"])
    .reset_index(drop=True)
)


# Indique si la variable appartient à la sélection métier de référence.
audit_candidates["selection_reference"] = audit_candidates.apply(
    lambda r: r["variable"]
    in FEATURE_BLOCKS_SELECTED_REFERENCE.get(r["famille"], []),
    axis=1,
)


# Variable retenue dans le référentiel final :
# - présente dans la base ;
# - admissible techniquement ;
# - validée dans la sélection métier.
audit_candidates["retenue_finale"] = (
    audit_candidates["selection_reference"]
    & audit_candidates["presente"]
    & audit_candidates["admissible_base"]
)


# Construction du dictionnaire final de variables retenues par famille.
# Ce bloc évite l'erreur FEATURE_BLOCKS_SELECTED non défini
# dans les cellules suivantes.
FEATURE_BLOCKS_SELECTED = {
    fam: audit_candidates.loc[
        (audit_candidates["famille"] == fam)
        & (audit_candidates["retenue_finale"]),
        "variable",
    ].tolist()
    for fam in FEATURE_BLOCKS_CANDIDATES.keys()
}


# Synthèse de contrôle par famille.
audit_summary = (
    audit_candidates
    .groupby("famille", dropna=False)
    .agg(
        nb_variables_reference=("variable", "nunique"),
        nb_presentes=("presente", "sum"),
        nb_admissibles=("admissible_base", "sum"),
        nb_selection_reference=("selection_reference", "sum"),
        nb_retenues_finales=("retenue_finale", "sum"),
        pct_manquant_median=("pct_manquant", "median"),
    )
    .reset_index()
)


display(audit_summary)

display(
    audit_candidates
    .sort_values(["famille", "retenue_finale", "pct_manquant"], ascending=[True, False, True])
    .head(80)
)

,famille,variable,presente,dtype,pct_manquant,nunique,is_numeric,is_leakage_or_target,is_geo_proxy,admissible_base,selection_reference,retenue_finale,decision
3,B1_bien,classe_surface,True,category,0.0000,8,False,False,False,True,True,True,retenue
18,B1_bien,has_dependance,True,float64,0.0000,2,True,False,False,True,True,True,retenue
12,B1_bien,is_appartement,True,int32,0.0000,2,True,False,False,True,True,True,retenue
11,B1_bien,is_maison,True,int32,0.0000,2,True,False,False,True,True,True,retenue
2,B1_bien,log_surface_reference,True,float64,0.0000,33488,True,False,False,True,True,True,retenue
...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,B6_taux_credit,taux_credit_lag3,True,Float64,0.0000,52,True,False,False,True,True,True,retenue
242,B6_taux_credit,taux_credit_moyen,True,Float64,0.0000,50,True,False,False,True,True,True,retenue
245,B6_taux_credit,taux_credit_roll3,True,float64,0.0000,56,True,False,False,True,True,True,retenue
243,B6_taux_credit,variation_taux_1m,True,Float64,0.0000,40,True,False,False,True,True,True,retenue


,famille,nb_candidates,nb_presentes,nb_retenues
0,B1_bien,22,17,17
1,B2_temps,25,18,15
2,B3_territoire,18,8,8
3,B4_socio_eco,23,14,12
4,B5_comparables,154,154,30
5,B6_taux_credit,7,7,7


Exports sélection : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\selection_features_scenarios\config
B5 candidates : 154
B5 retenues : 30
['commune_nb_ventes_passe', 'commune_type_nb_ventes_passe', 'iris_nb_ventes_passe', 'commune_prix_m2_med_passe', 'commune_type_prix_m2_med_passe', 'iris_prix_m2_med_passe', 'geo_1000m_12m_all_nb', 'geo_1000m_24m_all_nb', 'geo_300m_12m_all_nb', 'geo_300m_24m_all_nb', 'geo_500m_12m_all_nb', 'geo_500m_24m_all_nb', 'geo_1000m_24m_all_dist_med', 'geo_1000m_24m_all_prix_m2_iqr', 'geo_1000m_24m_all_prix_m2_med', 'geo_1000m_12m_all_dist_med', 'geo_1000m_12m_all_prix_m2_iqr', 'geo_1000m_12m_all_prix_m2_med', 'geo_500m_24m_all_dist_med', 'geo_500m_24m_all_prix_m2_iqr', 'geo_500m_24m_all_prix_m2_med', 'geo_500m_12m_all_dist_med', 'geo_500m_12m_all_prix_m2_iqr', 'geo_500m_12m_all_prix_m2_med', 'geo_300m_24m_all_dist_med', 'geo_300m_24m_all_prix_m2_iqr', 'geo_300m_24m_all_prix_m2_med', 'geo_300m_12m_all_dist_med', 'geo_300m_12m_al

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">

  <div style="color: #4b5563; font-size: 14px;">Scénarios de modélisation et mesure d’importance Les scénarios sont construits pour isoler les contributions successives des familles de variables. Le socle B1 décrit le bien ; les blocs B2 à B5 ajoutent respectivement le temps, le territoire, le socio-économique et les comparables ; le bloc B6 teste l’effet spécifique des taux. Les scénarios simples mesurent la contribution isolée d’une famille.  Les scénarios incrémentaux testent l’amélioration progressive du pouvoir explicatif.  Les variantes  _avec_taux  permettent de comparer chaque scénario avec et sans contexte de crédit.</div>
</div>

In [ ]:
 
# Scénarios progressifs et intermédiaires pour modélisation
 
# Objectif : mesurer la contribution marginale des familles de variables.
# Les scénarios sont volontairement lisibles :
# - un socle B1,
# - des apports isolés B2/B3/B4/B5,
# - des combinaisons incrémentales,
# - un scénario complet sélectionné.

SCENARIO_FEATURES = {}

def uniq_features(*blocks):
    """Concatène des listes de variables en conservant l'ordre et sans doublons."""
    merged = []
    for block in blocks:
        merged.extend(block or [])
    return list(dict.fromkeys(merged))

B1 = FEATURE_BLOCKS_SELECTED.get("B1_bien", [])
B2 = FEATURE_BLOCKS_SELECTED.get("B2_temps", [])
B3 = FEATURE_BLOCKS_SELECTED.get("B3_territoire", [])
B4 = FEATURE_BLOCKS_SELECTED.get("B4_socio_eco", [])
B5 = FEATURE_BLOCKS_SELECTED.get("B5_comparables", [])
B6 = FEATURE_BLOCKS_SELECTED.get("B6_taux_credit", features_taux_credit)

# Socle minimal : caractéristiques intrinsèques du bien.
SCENARIO_FEATURES["S_B1_only"] = uniq_features(B1)

# Apports marginaux isolés par famille autour du socle B1.
SCENARIO_FEATURES["S_B1_B2"] = uniq_features(B1, B2)
SCENARIO_FEATURES["S_B1_B3"] = uniq_features(B1, B3)
SCENARIO_FEATURES["S_B1_B4"] = uniq_features(B1, B4)
SCENARIO_FEATURES["S_B1_B5"] = uniq_features(B1, B5)

# Scénarios incrémentaux progressifs.
SCENARIO_FEATURES["S_B1_B2_B3"] = uniq_features(B1, B2, B3)
SCENARIO_FEATURES["S_B1_B2_B5"] = uniq_features(B1, B2, B5)
SCENARIO_FEATURES["S_B1_B2_B3_B4"] = uniq_features(B1, B2, B3, B4)
SCENARIO_FEATURES["S_FULL_SELECTED"] = uniq_features(B1, B2, B3, B4, B5)

# Scénarios analytiques optionnels : contribution pure par famille.
SCENARIO_FEATURES["S_B2_only"] = uniq_features(B2)
SCENARIO_FEATURES["S_B3_only"] = uniq_features(B3)
SCENARIO_FEATURES["S_B4_only"] = uniq_features(B4)
SCENARIO_FEATURES["S_B5_only"] = uniq_features(B5)


# Scénario analytique taux seul.
SCENARIO_FEATURES["S_B6_taux_only"] = uniq_features(B6)

# Déclinaisons avec taux des scénarios principaux.
for scenario_name, feature_list in list(SCENARIO_FEATURES.items()):
    if scenario_name.endswith("_avec_taux") or scenario_name == "S_B6_taux_only":
        continue

    SCENARIO_FEATURES[f"{scenario_name}_avec_taux"] = uniq_features(feature_list, B6)


scenario_metadata = {
    "S_B1_only": "Socle bien : contribution des caractéristiques intrinsèques.",
    "S_B1_B2": "Ajout du temps au socle bien.",
    "S_B1_B3": "Ajout isolé du territoire / marché au socle bien.",
    "S_B1_B4": "Ajout isolé du socio-économique au socle bien.",
    "S_B1_B5": "Ajout isolé des comparables au socle bien.",
    "S_B1_B2_B3": "Socle bien + temps + territoire.",
    "S_B1_B2_B5": "Socle bien + temps + comparables.",
    "S_B1_B2_B3_B4": "Socle complet hors comparables.",
    "S_FULL_SELECTED": "Toutes les variables sélectionnées B1 à B5.",
    "S_B2_only": "Lecture analytique pure du bloc temps.",
    "S_B3_only": "Lecture analytique pure du bloc territoire / marché.",
    "S_B4_only": "Lecture analytique pure du bloc socio-économique.",
    "S_B5_only": "Lecture analytique pure du bloc comparables.",
}



scenario_metadata["S_B6_taux_only"] = "Lecture analytique pure de la famille taux de crédit."

for scenario_name in list(SCENARIO_FEATURES.keys()):
    if scenario_name.endswith("_avec_taux"):
        base_name = scenario_name.replace("_avec_taux", "")
        scenario_metadata[scenario_name] = (
            scenario_metadata.get(base_name, base_name)
            + " + taux de crédit."
        )

with open(CONFIG_DIR / "scenario_features_v2.json", "w", encoding="utf-8") as f:
    json.dump(SCENARIO_FEATURES, f, ensure_ascii=False, indent=2)

with open(CONFIG_DIR / "scenario_metadata_v2.json", "w", encoding="utf-8") as f:
    json.dump(scenario_metadata, f, ensure_ascii=False, indent=2)

scenario_summary = pd.DataFrame([
    {
        "scenario": k,
        "objectif": scenario_metadata.get(k, ""),
        "nb_features": len(v),
        "integre_taux": any(feature in features_taux_credit for feature in v),
        "target_col": TARGET,
        "features": ", ".join(v),
    }
    for k, v in SCENARIO_FEATURES.items()
])

display(scenario_summary)
scenario_summary.to_csv(REPORTS_DIR / "scenario_features_v2.csv", index=False)

print("Scénarios exportés :", CONFIG_DIR / "scenario_features_v2.json")


,scenario,objectif,nb_features,integre_taux,target_col,features
0,S_B1_only,Socle bien : contribution des caractéristiques intrinsèques.,17,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
1,S_B1_B2,Ajout du temps au socle bien.,32,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
2,S_B1_B3,Ajout isolé du territoire / marché au socle bien.,25,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
3,S_B1_B4,Ajout isolé du socio-économique au socle bien.,29,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
4,S_B1_B5,Ajout isolé des comparables au socle bien.,47,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
5,S_B1_B2_B3,Socle bien + temps + territoire.,40,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
6,S_B1_B2_B5,Socle bien + temps + comparables.,62,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
7,S_B1_B2_B3_B4,Socle complet hors comparables.,52,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
8,S_FULL_SELECTED,Toutes les variables sélectionnées B1 à B5.,82,False,log_prix_m2,"classe_surface, has_dependance, is_appartement, is_maison, log_surface_reference, nb_dependances, pieces_info_disponible, segment_residentiel_surface, surface_reference, terrai..."
9,S_B2_only,Lecture analytique pure du bloc temps.,15,False,log_prix_m2,"annee, annee_mutation, mois_cos, mois_depuis_debut, mois_sin, trimestre, trimestre_mutation, score_momentum_lag1, score_risque_temporel_lag1, score_tension_temporelle_lag1, vol..."


Scénarios exportés : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\outputs\selection_features_scenarios\config\scenario_features_v2.json


In [ ]:
# Export
df_modelisation_scenarios_features = df.copy()

df_modelisation_scenarios_features.to_parquet(OUTPUT_PATH, index=False)

df_scenarios_features_metadata = pd.DataFrame([
    {
        "scenario": scenario,
        "nb_features": len(features),
        "integre_taux": any(feature in features_taux_credit for feature in features),
        "target_col": TARGET,
    }
    for scenario, features in SCENARIO_FEATURES.items()
])

df_scenarios_features_metadata.to_parquet(OUTPUT_METADATA_PATH, index=False)

with open(OUTPUT_FEATURE_LISTS_PATH, "w", encoding="utf-8") as f:
    json.dump(SCENARIO_FEATURES, f, ensure_ascii=False, indent=2)

df_scenarios_features_metadata.to_csv(REPORTS_DIR / "df_scenarios_features_metadata.csv", index=False)

print("Base scénarios exportée :", OUTPUT_PATH)
print("Métadonnées scénarios exportées :", OUTPUT_METADATA_PATH)
print("Dictionnaire scénarios exporté :", OUTPUT_FEATURE_LISTS_PATH)
print("Nombre de scénarios :", len(SCENARIO_FEATURES))
print("Scénarios avec taux :", df_scenarios_features_metadata["integre_taux"].sum())

Base scénarios exportée : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_modelisation_scenarios_features.parquet
Métadonnées scénarios exportées : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\df_scenarios_features_metadata.parquet
Dictionnaire scénarios exporté : C:\Users\club_\OneDrive\13_DOCUMENT\SYSTEME_AIDE_DECISION_IMMOBILIERE\data\processed\dict_scenarios_features.json
Nombre de scénarios : 27
Scénarios avec taux : 14


<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Conclusion</h3>
  <div style="color: #4b5563; font-size: 14px;">Synthèse opérationnelle Le notebook 10 clôt la phase de préparation des variables pour la modélisation. Les familles de features sont auditées, les variables à risque sont écartées, les scénarios sont formalisés et les fichiers nécessaires à la comparaison des modèles sont produits. Résultat clé : 27 scénarios disponibles pour comparer les apports des familles de variables.  Limite : l’admissibilité statistique ne remplace pas l’évaluation prédictive ; elle prépare seulement un cadre robuste de test.  Sortie transmise : base et référentiels de scénarios pour le notebook de modélisation.</div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #d9e2ec; border-left: 4px solid #1f4e79; border-radius: 6px; padding: 14px 18px; margin: 16px 0; color: #1f2933; font-family: Arial, sans-serif; line-height: 1.55;">
  <h3 style="margin: 0 0 8px 0; color: #1f4e79; font-size: 18px;">Conclusion</h3>
  <div style="color: #4b5563; font-size: 14px;">Synthèse du notebook 10 La préparation est désormais suffisamment structurée pour passer à l’évaluation prédictive. Le notebook suivant pourra comparer les scénarios sur des métriques de performance, de robustesse et d’interprétabilité.</div>
</div>